# Sudoku RL local notebook

This notebook loads `Qwen/Qwen3-0.6B` from Hugging Face with `transformers` and runs the same Sudoku episode loop as `inference.py`, but fully local for model generation.

If you already have the dependencies installed, you can skip the install cell.

## Kernel check

If `pip install torch` succeeded in a terminal but `import torch` fails here, the notebook is almost certainly using a different Python environment.

Run the next cell first, then install packages with `%pip` inside this notebook so they go into the active kernel.

In [1]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Working directory:", Path.cwd())

try:
    import torch
    print("torch already installed:", torch.__version__)
except ModuleNotFoundError:
    print("torch is not installed in this notebook kernel.")
    print("Run the next install cell with %pip, then restart the kernel.")

Python executable: /home/zeus/miniconda3/envs/cloudspace/bin/python
Python version: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:09:17) [GCC 11.2.0]
Working directory: /teamspace/studios/this_studio/sudoku_rl
torch already installed: 2.8.0+cu128


In [2]:
# %pip install -q torch transformers accelerate openenv-core[core]

In [9]:
import asyncio
import json
import os
import re
import textwrap
from threading import Thread
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from openenv.core.containers.runtime.providers import LocalDockerProvider

from sudoku_rl import SudokuRlAction, SudokuRlEnv

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3-0.6B")
IMAGE_NAME = os.getenv("IMAGE_NAME", "sudoku_rl:latest")
BASE_URL = os.getenv("SUDOKU_BASE_URL", "")
EMPTY_BOXES = int(os.getenv("EMPTY_BOXES", "35"))
MAX_STEPS = int(os.getenv("MAX_STEPS", "60"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "2048"))
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.2"))
TOP_P = float(os.getenv("TOP_P", "0.8"))
CONNECT_TIMEOUT_S = float(os.getenv("CONNECT_TIMEOUT_S", "60"))
MESSAGE_TIMEOUT_S = float(os.getenv("MESSAGE_TIMEOUT_S", "1800"))
DOCKER_WAIT_TIMEOUT_S = float(os.getenv("DOCKER_WAIT_TIMEOUT_S", "180"))

SYSTEM_PROMPT = textwrap.dedent(
    """
    You are solving a Sudoku puzzle one move at a time.

    Each step, you are given:
    - the current board
    - a knowledge snapshot built from that board
    - recent action and environment history
    - the last move and the environment response to that move

    You must use the knowledge snapshot as your primary working memory.
    The knowledge snapshot already summarizes, for every row, column, and 3x3 box:
    - which cells are empty
    - which values are missing

    Required solving procedure:
    1. Read the last move and the last environment response first.
    2. If the last move was invalid, do not repeat it.
    3. Read the knowledge snapshot next.
    4. Before choosing any move, search for a forced move in this priority order:
       a. a row with exactly one empty cell
       b. a column with exactly one empty cell
       c. a 3x3 box with exactly one empty cell
    5. If a forced move exists, choose that forced move immediately.
    6. If no forced move exists, pick an empty cell from the snapshot.
    7. Check row possibilities first.
    8. If the row alone gives exactly one possible value, stop there and use that answer.
    9. Only if the row does not give a single answer, check the column.
    10. If the column then gives exactly one possible value, stop there and use that answer.
    11. Only if both row and column are still not enough, check the 3x3 box.
    12. After combining row, column, and box constraints, choose a move only when there is exactly one valid value.
    13. If multiple values are still possible, do not guess. Move to another empty cell.

    Priority rules:
    - Prefer cells where the row already determines a single value.
    - If a row has exactly one empty cell, solve that row first.
    - If no row-only answer exists, prefer a column-based forced answer.
    - If a column has exactly one empty cell, solve that column before considering non-forced moves.
    - If neither row nor column gives a forced answer, use the 3x3 box as the final filter.
    - If a box has exactly one empty cell, solve that box before considering non-forced moves.
    - Do not choose cells from the original fixed puzzle.
    - Never repeat an action that already produced `invalid_move`.
    - If the last move was invalid, you must choose a different row, column, or value.
    - Treat the environment message as a hard constraint. For example, if the message says `number in row`, do not place that value in that row again.

    Reasoning style:
    - Start by checking your last move and the environment response.
    - If the last move was invalid, explicitly avoid repeating that move.
    - Keep reasoning grounded in the knowledge snapshot.
    - First mention any forced move discovered from rows, columns, or boxes with one empty cell.
    - Refer to row, column, and box empties/missing values from the snapshot before selecting a move.
    - Use short, structured reasoning.
    - Treat the knowledge snapshot as rebuilt after every move.

    Output rules:
    - You may include reasoning.
    - Do not include markdown or code fences.
    - Output exactly one final JSON object.
    - The final JSON must be the last structured object in the response.
    - The final JSON must have exactly this schema:
      {"row": <1-9>, "column": <1-9>, "value": <1-9>}
    """
).strip()

if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {
        "torch_dtype": torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        "device_map": "auto",
    }
elif torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"torch_dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"torch_dtype": torch.float32}

print(f"Loading {MODEL_NAME} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True, **model_kwargs)
if "device_map" not in model_kwargs:
    model = model.to(device)
model.eval()
print("Model loaded.")

Loading Qwen/Qwen3-0.6B on cpu...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model loaded.


In [10]:
def build_user_prompt(step: int, observation: Any, history: list[str]) -> str:
    history_block = "\n".join(history[-6:]) if history else "None"
    invalid_block = observation.invalid_cells if observation.invalid_cells else "[]"
    knowledge_snapshot = build_knowledge_snapshot(observation)
    forbidden_actions = [entry for entry in history[-6:] if "status=invalid_move" in entry]
    forbidden_block = "\n".join(forbidden_actions) if forbidden_actions else "None"
    last_move_block = history[-1] if history else "None"
    return textwrap.dedent(
        f"""
        Step: {step}
        Status: {observation.status}
        Score: {observation.score}
        Score delta from last move: {observation.score_delta}
        Moves attempted: {observation.moves}
        Mistakes: {observation.mistakes}
        Empty cells remaining: {observation.empty_cells_remaining}
        Incorrect cells: {observation.incorrect_cells}
        Invalid cells (0-based [row, col]): {invalid_block}
        Last environment message: {observation.message}

        Current board:
        {observation.board_text}

        Knowledge snapshot built from the current board:
        {knowledge_snapshot}

        Last move and its outcome:
        {last_move_block}

        Recent invalid actions that must not be repeated:
        {forbidden_block}

        Previous recent actions and responses:
        {history_block}

        First inspect the last move and the environment response.
        If the last move was invalid, do not repeat it.
        Then use the knowledge snapshot to understand empty cells in each row, column, and 3x3 box.
        Look for a forced move first: a row, column, or box with exactly one empty cell.
        If a forced move exists, take it immediately.
        Never repeat any action listed in the invalid-action block.
        If the last environment message says `number in row`, do not place that value in that row again.
        If the last environment message says `number in column`, do not place that value in that column again.
        If the last environment message says `number in box`, do not place that value in that 3x3 box again.
        Then use that knowledge to solve the next move.
        After each step, assume this knowledge snapshot is rebuilt from the updated board.
        Return the next move as JSON.
        """
    ).strip()


def _extract_digit_1_to_9(value: Any) -> int | None:
    if isinstance(value, int):
        return value if 1 <= value <= 9 else None

    if isinstance(value, str):
        match = re.search(r"\b([1-9])\b", value)
        if match:
            return int(match.group(1))

    return None


def format_board(board: list[list[int]]) -> str:
    rows: list[str] = []
    for row_index, row in enumerate(board):
        cells = [str(value) if value != 0 else "." for value in row]
        rows.append(" ".join(cells[:3]) + " | " + " ".join(cells[3:6]) + " | " + " ".join(cells[6:9]))
        if row_index in {2, 5}:
            rows.append("---------------------")
    return "\n".join(rows)


def _missing_values(values: list[int]) -> list[int]:
    present = {value for value in values if value != 0}
    return [value for value in range(1, 10) if value not in present]


def build_knowledge_snapshot(observation: Any) -> str:
    board = observation.board
    lines: list[str] = []

    lines.append("Row knowledge:")
    for row_index, row in enumerate(board, start=1):
        empty_columns = [column_index for column_index, value in enumerate(row, start=1) if value == 0]
        missing = _missing_values(row)
        lines.append(f"- row {row_index}: empty_columns={empty_columns or '[]'} missing={missing or '[]'}")

    lines.append("Column knowledge:")
    for column_index in range(9):
        column_values = [board[row_index][column_index] for row_index in range(9)]
        empty_rows = [row_index + 1 for row_index, value in enumerate(column_values) if value == 0]
        missing = _missing_values(column_values)
        lines.append(f"- column {column_index + 1}: empty_rows={empty_rows or '[]'} missing={missing or '[]'}")

    lines.append("Box knowledge:")
    box_number = 1
    for box_row in range(0, 9, 3):
        for box_column in range(0, 9, 3):
            cells: list[int] = []
            empty_cells: list[str] = []
            for row_index in range(box_row, box_row + 3):
                for column_index in range(box_column, box_column + 3):
                    value = board[row_index][column_index]
                    cells.append(value)
                    if value == 0:
                        empty_cells.append(f"r{row_index + 1}c{column_index + 1}")
            missing = _missing_values(cells)
            lines.append(f"- box {box_number}: empty_cells={empty_cells or '[]'} missing={missing or '[]'}")
            box_number += 1

    return "\n".join(lines)


def parse_action(text: str) -> SudokuRlAction | None:
    cleaned = (text or "").strip()
    if not cleaned:
        return None

    if "</think>" in cleaned:
        cleaned = cleaned.split("</think>", 1)[1].strip()

    candidates: list[dict[str, Any]] = []
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            candidates.append(parsed)
    except json.JSONDecodeError:
        pass

    for match in re.finditer(r"\{[^{}]+\}", cleaned, flags=re.DOTALL):
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            candidates.append(parsed)

    row_match = re.search(r'"?row"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    column_match = re.search(r'"?(?:column|col)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    value_match = re.search(r'"?(?:value|number)"?\s*[:=]\s*([1-9])', cleaned, flags=re.IGNORECASE)
    if row_match and column_match and value_match:
        candidates.append(
            {
                "row": int(row_match.group(1)),
                "column": int(column_match.group(1)),
                "value": int(value_match.group(1)),
            }
        )

    for candidate in candidates:
        try:
            row = _extract_digit_1_to_9(candidate["row"])
            column = _extract_digit_1_to_9(candidate["column"])
            value = _extract_digit_1_to_9(candidate.get("value", candidate.get("number")))
        except (KeyError, TypeError, ValueError):
            continue
        if row is not None and column is not None and value is not None:
            return SudokuRlAction(row=row, column=column, value=value)

    return None


def candidates_for_cell(board: list[list[int]], row_index: int, column_index: int) -> list[int]:
    if board[row_index][column_index] != 0:
        return []

    used_values = set(board[row_index])
    used_values.update(board[r][column_index] for r in range(9))

    box_row = (row_index // 3) * 3
    box_column = (column_index // 3) * 3
    for sub_row in range(box_row, box_row + 3):
        for sub_column in range(box_column, box_column + 3):
            used_values.add(board[sub_row][sub_column])

    return [value for value in range(1, 10) if value not in used_values]


def heuristic_action(observation: Any) -> SudokuRlAction:
    board = [row[:] for row in observation.board]
    invalid_targets = {
        (int(row_index), int(column_index))
        for row_index, column_index in observation.invalid_cells
        if 0 <= row_index < 9 and 0 <= column_index < 9
    }

    for row_index, column_index in invalid_targets:
        board[row_index][column_index] = 0

    target_cells: list[tuple[int, int]] = list(invalid_targets)
    for row_index in range(9):
        for column_index in range(9):
            if observation.initial_puzzle[row_index][column_index] != 0:
                continue
            if board[row_index][column_index] == 0 and (row_index, column_index) not in invalid_targets:
                target_cells.append((row_index, column_index))

    if not target_cells:
        for row_index in range(9):
            for column_index in range(9):
                if observation.initial_puzzle[row_index][column_index] == 0:
                    target_cells.append((row_index, column_index))

    for row_index, column_index in target_cells:
        values = candidates_for_cell(board, row_index, column_index)
        if values:
            return SudokuRlAction(row=row_index + 1, column=column_index + 1, value=values[0])

    row_index, column_index = target_cells[0] if target_cells else (0, 0)
    return SudokuRlAction(row=row_index + 1, column=column_index + 1, value=1)


def is_action_valid(observation: Any, action: SudokuRlAction) -> tuple[bool, str | None]:
    row_index = action.row - 1
    column_index = action.column - 1
    value = action.value

    if not (0 <= row_index < 9 and 0 <= column_index < 9 and 1 <= value <= 9):
        return False, "out_of_range"

    if observation.initial_puzzle[row_index][column_index] != 0:
        return False, "fixed_cell"

    board = [row[:] for row in observation.board]
    board[row_index][column_index] = 0

    if value in board[row_index]:
        return False, "number_in_row"

    if value in [board[r][column_index] for r in range(9)]:
        return False, "number_in_column"

    box_row = (row_index // 3) * 3
    box_column = (column_index // 3) * 3
    for sub_row in range(box_row, box_row + 3):
        for sub_column in range(box_column, box_column + 3):
            if board[sub_row][sub_column] == value:
                return False, "number_in_box"

    return True, None


def choose_guarded_action(
    observation: Any,
    action: SudokuRlAction,
    history: list[str],
) -> tuple[SudokuRlAction, str | None, bool]:
    action_signature = f"row:{action.row}, column:{action.column}, value:{action.value}"
    repeated_invalid = any(action_signature in entry and "status=invalid_move" in entry for entry in history[-10:])
    if repeated_invalid:
        return heuristic_action(observation), "repeated_invalid_action", True

    is_valid, local_error = is_action_valid(observation, action)
    if is_valid:
        return action, None, False

    return heuristic_action(observation), local_error, True


In [11]:
def generate_model_text(user_prompt: str, verbose: bool = False) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        try:
            model_inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        except TypeError:
            model_inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
    else:
        prompt_text = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
        model_inputs = tokenizer(prompt_text, return_tensors="pt")
    if "device_map" in model_kwargs:
        target_device = next(model.parameters()).device
        model_inputs = {key: value.to(target_device) for key, value in model_inputs.items()}
    else:
        model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

    generation_kwargs = {
        **model_inputs,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
        "temperature": TEMPERATURE if TEMPERATURE > 0 else None,
        "top_p": TOP_P if TEMPERATURE > 0 else None,
        "pad_token_id": tokenizer.eos_token_id,
    }
    generation_kwargs = {k: v for k, v in generation_kwargs.items() if v is not None}

    if verbose:
        streamer = TextIteratorStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=False,
        )
        streamed_kwargs = {
            **generation_kwargs,
            "streamer": streamer,
        }
        print("[GENERATED_TOKENS]")
        worker = Thread(target=model.generate, kwargs=streamed_kwargs)
        worker.start()
        streamed_chunks: list[str] = []
        for chunk in streamer:
            streamed_chunks.append(chunk)
            print(chunk, end="", flush=True)
        worker.join()
        print()
        text = "".join(streamed_chunks).strip()
    else:
        with torch.no_grad():
            outputs = model.generate(
                **generation_kwargs,
                return_dict_in_generate=True,
            )

        generated_ids = outputs.sequences[0][model_inputs["input_ids"].shape[-1]:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    if verbose:
        think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
        if think_match:
            print("[MODEL_REASONING]")
            print(think_match.group(1).strip())
        print("[MODEL_OUTPUT]")
        print(text)
    return text


def get_local_action(step: int, observation: Any, history: list[str], verbose: bool = False) -> tuple[SudokuRlAction, str, str | None, str]:
    user_prompt = build_user_prompt(step, observation, history)
    try:
        text = generate_model_text(user_prompt, verbose=verbose)
        action = parse_action(text)
        if action is not None:
            return action, "model", None, text
        return heuristic_action(observation), "heuristic_fallback", f"unparseable_model_output={text!r}", text
    except Exception as exc:
        return heuristic_action(observation), "heuristic_fallback", str(exc), ""


async def get_local_action_async(step: int, observation: Any, history: list[str], verbose: bool = False) -> tuple[SudokuRlAction, str, str | None, str]:
    return await asyncio.to_thread(get_local_action, step, observation, history, verbose)


In [12]:
async def create_env():
    if BASE_URL:
        client = SudokuRlEnv(
            base_url=BASE_URL,
            connect_timeout_s=CONNECT_TIMEOUT_S,
            message_timeout_s=MESSAGE_TIMEOUT_S,
        )
        return await client.connect()
    provider = LocalDockerProvider()
    base_url = provider.start_container(IMAGE_NAME)
    provider.wait_for_ready(base_url, timeout_s=DOCKER_WAIT_TIMEOUT_S)
    client = SudokuRlEnv(
        base_url=base_url,
        connect_timeout_s=CONNECT_TIMEOUT_S,
        message_timeout_s=MESSAGE_TIMEOUT_S,
        provider=provider,
    )
    return await client.connect()


async def run_episode(empty_boxes: int = EMPTY_BOXES, max_steps: int = MAX_STEPS, verbose: bool = False):
    env = await create_env()
    history: list[str] = []
    rewards: list[float] = []
    final_status = "unknown"
    final_score_raw = 0

    try:
        result = await env.reset(empty_boxes=empty_boxes)
        observation = result.observation
        print("[START]", f"empty_boxes={empty_boxes}", f"model={MODEL_NAME}")
        print(
            "[TIMEOUTS]",
            f"connect_timeout_s={CONNECT_TIMEOUT_S}",
            f"message_timeout_s={MESSAGE_TIMEOUT_S}",
            f"docker_wait_timeout_s={DOCKER_WAIT_TIMEOUT_S}",
        )
        print("[ORIGINAL_SUDOKU]")
        print(format_board(observation.initial_puzzle))
        print("[CURRENT_STATUS]", observation.status)
        print("[CURRENT_BOARD]")
        print(observation.board_text)
        print("[KNOWLEDGE_SNAPSHOT]")
        print(build_knowledge_snapshot(observation))

        for step in range(1, max_steps + 1):
            if result.done:
                break

            action, source, error, raw_text = await get_local_action_async(
                step,
                observation,
                history,
                verbose=verbose,
            )
            action, guard_error, used_guard = choose_guarded_action(observation, action, history)
            if used_guard:
                source = "local_guard_fallback"
                error = guard_error
            result = await env.step(action)
            observation = result.observation

            reward = float(result.reward or 0.0)
            rewards.append(reward)
            final_status = observation.status
            final_score_raw = observation.score

            print(
                "[STEP]",
                f"step={step}",
                f"row={action.row}",
                f"column={action.column}",
                f"value={action.value}",
                f"reward={reward:.2f}",
                f"score={observation.score}",
                f"status={observation.status}",
                f"source={source}",
                f"error={error or 'null'}",
            )
            print("[CURRENT_STATUS]", observation.status)
            print("[ORIGINAL_SUDOKU]")
            print(format_board(observation.initial_puzzle))
            print("[CURRENT_BOARD]")
            print(observation.board_text)
            print("[KNOWLEDGE_SNAPSHOT]")
            print(build_knowledge_snapshot(observation))
            print("[ENV_MESSAGE]", observation.message)

            history.append(
                f"step={step} raw={raw_text!r} action={{row:{action.row}, column:{action.column}, value:{action.value}}} "
                f"reward={reward:+.2f} status={observation.status} message={observation.message}"
            )

            if result.done:
                break

        max_possible_score = max(observation.empty_boxes * max(observation.score_step, 1), 1)
        normalized_score = max(min(final_score_raw / max_possible_score, 1.0), 0.0)
        success = final_status == "solved"
        completion_ratio = 1.0 - (observation.empty_cells_remaining / max(observation.empty_boxes, 1))
        completion_ratio = max(min(completion_ratio, 1.0), 0.0)
        if not success:
            normalized_score = min(normalized_score, 0.99)
        print("[END]", f"success={success}", f"score={normalized_score:.3f}", f"raw_score={final_score_raw}")
        return {
            "success": success,
            "normalized_score": normalized_score,
            "completion_ratio": completion_ratio,
            "empty_cells_remaining": observation.empty_cells_remaining,
            "raw_score": final_score_raw,
            "final_status": final_status,
            "rewards": rewards,
            "history": history,
            "board_text": observation.board_text,
            "verbose": verbose,
        }
    finally:
        try:
            await env.close()
        except Exception as exc:
            print(f"[CLEANUP_WARNING] {type(exc).__name__}: {exc}")


In [13]:
result = await run_episode()
result

[START] empty_boxes=35 model=Qwen/Qwen3-0.6B
[ORIGINAL_SUDOKU]
. . 5 | 1 . 4 | . 3 .
1 7 2 | . 6 5 | 9 . .
. . 4 | 2 . 9 | 6 5 1
---------------------
. . 8 | . . 7 | . . 3
. 9 . | . . . | 5 1 7
. . . | . 3 . | 8 . .
---------------------
8 5 . | 4 . 2 | 3 7 .
6 . 9 | 7 5 . | 1 8 4
4 3 7 | 6 1 8 | 2 9 .
[CURRENT_STATUS] ready
[CURRENT_BOARD]
. . 5 | 1 . 4 | . 3 .
1 7 2 | . 6 5 | 9 . .
. . 4 | 2 . 9 | 6 5 1
---------------------
. . 8 | . . 7 | . . 3
. 9 . | . . . | 5 1 7
. . . | . 3 . | 8 . .
---------------------
8 5 . | 4 . 2 | 3 7 .
6 . 9 | 7 5 . | 1 8 4
4 3 7 | 6 1 8 | 2 9 .
[MODEL_OUTPUT]
<think>
Okay, let's see. I need to find a cell to fill in the Sudoku puzzle. The current board is given, and I have to choose one cell and fill it with a value from 1-9. The rules say I should evaluate possible values for each cell and check if there's only one possible value. If there's only one, I can update it. Otherwise, I need to try other cells.

Looking at the current board, I need to chec

TimeoutExpired: Command '['docker', 'stop', '4e9b954a0710624dc8bc45f4a384cb54f780f4f728605708cea73f658c12ad8f']' timed out after 10 seconds